In [11]:
import pandas as pd

In [12]:
# Đọc dữ liệu từ file CSV
df = pd.read_csv('../data/raw/Global_Cybersecurity_Threats_2015-2024.csv')

# Xem tổng quan dữ liệu
print(df.info())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3060 entries, 0 to 3059
Data columns (total 10 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Country                              3018 non-null   object 
 1   Year                                 2970 non-null   float64
 2   Attack Type                          2978 non-null   object 
 3   Target Industry                      2947 non-null   object 
 4   Financial Loss (in Million $)        2996 non-null   float64
 5   Number of Affected Users             3013 non-null   float64
 6   Attack Source                        2933 non-null   object 
 7   Security Vulnerability Type          2913 non-null   object 
 8   Defense Mechanism Used               2977 non-null   object 
 9   Incident Resolution Time (in Hours)  2910 non-null   float64
dtypes: float64(4), object(6)
memory usage: 239.2+ KB
None
        Country    Year        Attack Type

In [13]:
# ======================= 1. XỬ LÝ DỮ LIỆU TRÙNG LẶP =======================

# Kiểm tra số lượng bản ghi trùng lặp
duplicates_before = df.duplicated().sum()
print(f"Số bản ghi trùng lặp trước khi xử lý: {duplicates_before}")

# Xoá bản ghi trùng lặp (nếu có)
df.drop_duplicates(inplace=True)

Số bản ghi trùng lặp trước khi xử lý: 60


In [14]:
# ======================= 2. XỬ LÝ DỮ LIỆU THIẾU ==========================

# Kiểm tra số lượng giá trị thiếu theo từng cột
missing_values_before = df.isnull().sum()
print("\nSố lượng giá trị thiếu trước khi xử lý:")
print(missing_values_before)

# Các cột phân loại (chuỗi): thay thế giá trị thiếu bằng giá trị phổ biến nhất (mode)
categorical_cols = ['Country', 'Attack Type', 'Target Industry', 'Attack Source', 
                   'Security Vulnerability Type', 'Defense Mechanism Used']
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Làm sạch khoảng trắng đầu/cuối chuỗi trong các cột dạng object
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()

# Các cột số: thay giá trị thiếu bằng median (trung vị)
numeric_cols = ['Financial Loss (in Million $)', 'Number of Affected Users', 'Incident Resolution Time (in Hours)']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')  # ép về số, lỗi sẽ thành NaN
    df[col] = df[col].fillna(df[col].median())

# Cột Year cần xử lý đặc biệt
df['Year'] = df['Year'].astype(str).str.strip()  # bỏ khoảng trắng
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')  # chuyển về số, lỗi thành NaN
df['Year'] = df['Year'].fillna(df['Year'].mode()[0])  # điền giá trị phổ biến nhất




Số lượng giá trị thiếu trước khi xử lý:
Country                                 41
Year                                    88
Attack Type                             81
Target Industry                        112
Financial Loss (in Million $)           64
Number of Affected Users                46
Attack Source                          123
Security Vulnerability Type            144
Defense Mechanism Used                  81
Incident Resolution Time (in Hours)    146
dtype: int64


In [15]:
# ======================= 3. CHUẨN HÓA GIÁ TRỊ & KIỂU DỮ LIỆU =================

# Chuẩn hóa chữ hoa/thường: Viết hoa chữ cái đầu cho dữ liệu dạng phân loại
for col in categorical_cols:
    df[col] = df[col].str.title()

# Kiểu dữ liệu năm đảm bảo là số nguyên
df['Year'] = df['Year'].astype(int)

In [16]:
# ======================= 4. KIỂM TRA & SỬA GIÁ TRỊ KHÔNG HỢP LỆ =================

# Kiểm tra các năm ngoài khoảng 2015–2024
invalid_years = df[~df['Year'].between(2015, 2024)]
print(f"\nSố dòng có giá trị 'Year' không hợp lệ: {len(invalid_years)}")

# Ép giá trị năm về khoảng hợp lệ
df['Year'] = df['Year'].clip(2015, 2024)

# Kiểm tra các giá trị âm trong cột số
negative_values = df[(df[numeric_cols] < 0).any(axis=1)]
print(f"Số dòng có giá trị âm trong các cột số: {len(negative_values)}")

# Loại bỏ giá trị âm bằng cách ép về 0 nếu nhỏ hơn 0
for col in numeric_cols:
    df[col] = df[col].clip(lower=0)


Số dòng có giá trị 'Year' không hợp lệ: 0
Số dòng có giá trị âm trong các cột số: 0


In [17]:
# ======================= 5. KIỂM TRA LẠI DỮ LIỆU =======================

# Kiểm tra còn thiếu dữ liệu không
missing_values_after = df.isnull().sum()
print("\nSố lượng giá trị thiếu sau khi xử lý:")
print(missing_values_after)


Số lượng giá trị thiếu sau khi xử lý:
Country                                0
Year                                   0
Attack Type                            0
Target Industry                        0
Financial Loss (in Million $)          0
Number of Affected Users               0
Attack Source                          0
Security Vulnerability Type            0
Defense Mechanism Used                 0
Incident Resolution Time (in Hours)    0
dtype: int64


In [18]:
# ======================= 6. LƯU FILE ĐÃ LÀM SẠCH =======================

df.to_csv('../data/processed/Cleaned_Global_Cybersecurity_Threats_2015-2024.csv', index=False)
print("\n✅ File dữ liệu đã được làm sạch và lưu thành công!")


✅ File dữ liệu đã được làm sạch và lưu thành công!
